In [2]:
import pandas as pd
import sqlite3

# 1. Téléchargement direct des vraies données depuis le dépot officiel
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
print("Téléchargement des réelles en cours (500k+ lignes)...")
df_real = pd.read_excel(url)

# 2. Aperçu rapide pour vérifier la structure
print("\nDonnées chargées avec succés !")
print(df_real.head())

# 3. Création d'une nouvelle base SQL pour notre nettoyage
conn = sqlite3.connect("ecom_marketing_real.db")
df_real.to_sql("retail_brut", conn, if_exists="replace", index=False)
conn.close()
print("\nBase 'ecom_marketing_real.db créée avec la table 'retail_brut'.")

Téléchargement des réelles en cours (500k+ lignes)...

Données chargées avec succés !
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  

Base 'ecom_marketing_real.db créée avec la table 'retail_brut'.


In [3]:
%load_ext sql
%sql sqlite:///ecom_marketing_real.db

In [4]:
%%sql
SELECT COUNT(*) FROM retail_brut
WHERE CustomerID IS NULL

 * sqlite:///ecom_marketing_real.db
Done.


COUNT(*)
135080


In [5]:
%%sql
DELETE FROM retail_brut
WHERE CustomerID IS NULL
   OR Quantity <= 0
   OR UnitPrice <= 0

 * sqlite:///ecom_marketing_real.db
144025 rows affected.


[]

In [6]:
%%sql
SELECT COUNT (*) FROM retail_brut

 * sqlite:///ecom_marketing_real.db
Done.


COUNT (*)
397884


In [7]:
%%sql
SELECT COUNT(*) FROM (SELECT DISTINCT * FROM retail_brut)

 * sqlite:///ecom_marketing_real.db
Done.


COUNT(*)
392692


In [8]:
%%sql
CREATE TABLE retail_propre AS SELECT DISTINCT * FROM retail_brut


 * sqlite:///ecom_marketing_real.db
Done.


[]

In [9]:
%%sql
DROP TABLE retail_brut

 * sqlite:///ecom_marketing_real.db
Done.


[]

In [10]:
%%sql
ALTER TABLE retail_propre RENAME TO retail_brut

 * sqlite:///ecom_marketing_real.db
Done.


[]

In [11]:
%%sql
SELECT Country, SUM(Quantity*UnitPrice) AS chiffre_affaire FROM retail_brut
GROUP BY Country
ORDER BY chiffre_affaire DESC
LIMIT 10

 * sqlite:///ecom_marketing_real.db
Done.


Country,chiffre_affaire
United Kingdom,7285024.644
Netherlands,285446.34
EIRE,265262.46
Germany,228678.4
France,208934.31
Australia,138453.81
Spain,61558.56
Switzerland,56443.95
Belgium,41196.34
Sweden,38367.83


In [12]:
%%sql
SELECT Description, SUM(quantity) FROM retail_brut
GROUP BY Description
ORDER BY SUM (quantity) DESC
LIMIT 10

 * sqlite:///ecom_marketing_real.db
Done.


Description,SUM(quantity)
"PAPER CRAFT , LITTLE BIRDIE",80995
MEDIUM CERAMIC TOP STORAGE JAR,77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS,54319
JUMBO BAG RED RETROSPOT,46078
WHITE HANGING HEART T-LIGHT HOLDER,36706
ASSORTED COLOUR BIRD ORNAMENT,35263
PACK OF 72 RETROSPOT CAKE CASES,33670
POPCORN HOLDER,30919
RABBIT NIGHT LIGHT,27153
MINI PAINT SET VINTAGE,26076


In [13]:
%%sql
SELECT AVG(chiffre_affaire) 
   FROM  (SELECT InvoiceNo, SUM(quantity*UnitPrice) AS chiffre_affaire FROM retail_brut
           GROUP BY InvoiceNo) AS  CA_par_commande


 * sqlite:///ecom_marketing_real.db
Done.


AVG(chiffre_affaire)
479.56016047917115


In [14]:
%%sql
SELECT strftime('%Y-%m', InvoiceDate) as Mois, SUM(quantity*UnitPrice) AS CA FROM retail_brut
GROUP BY Mois
ORDER BY CA DESC

 * sqlite:///ecom_marketing_real.db
Done.


Mois,CA
2011-11,1156205.61
2011-10,1035642.45
2011-09,950690.202
2011-05,677355.15
2011-06,660046.05
2011-08,644051.04
2011-07,598962.901
2011-03,594081.76
2010-12,570422.73
2011-01,568101.31


In [15]:
import pandas as pd
import sqlite3 

conn = sqlite3.connect("ecom_marketing_real.db") 

df_ca_pays = pd.read_sql_query("""
     SELECT Country, SUM(Quantity*UnitPrice) AS CA
     FROM retail_brut
     GROUP BY Country
     ORDER BY CA DESC
""", conn)

df_ca_pays.to_csv('ca_par_pays.csv',index=False)

In [16]:
df_ca_mois = pd.read_sql_query("""
     SELECT strftime('%Y-%m', InvoiceDate) as Mois, SUM(quantity*UnitPrice) AS CA FROM retail_brut
     GROUP BY Mois
     ORDER BY CA DESC
""", conn)

df_ca_mois.to_csv('ca_par_mois', index=False)

In [17]:
df_top_produits = pd.read_sql_query("""
       SELECT Description, SUM(quantity) FROM retail_brut
       GROUP BY Description
       ORDER BY SUM (quantity) DESC
""", conn)

df_top_produits.to_csv('top_produits', index=False)

In [18]:
%%sql
CREATE TABLE dim_clients AS
  SELECT DISTINCT customerID, Country
  FROM retail_brut

 * sqlite:///ecom_marketing_real.db
(sqlite3.OperationalError) table dim_clients already exists
[SQL: CREATE TABLE dim_clients AS
  SELECT DISTINCT customerID, Country
  FROM retail_brut]
(Background on this error at: https://sqlalche.me/e/14/e3q8)


In [19]:
%%sql
CREATE TABLE dim_produits AS
  SELECT DISTINCT StockCode, Description
  FROM retail_brut

 * sqlite:///ecom_marketing_real.db
(sqlite3.OperationalError) table dim_produits already exists
[SQL: CREATE TABLE dim_produits AS
  SELECT DISTINCT StockCode, Description
  FROM retail_brut]
(Background on this error at: https://sqlalche.me/e/14/e3q8)


In [20]:
%%sql
CREATE TABLE faits_ventes AS
  SELECT 
     InvoiceNo,
     StockCode,
     CustomerID,
     Quantity,
     UnitPrice,
     Date(InvoiceDate) AS DateVente
  FROM retail_brut

 * sqlite:///ecom_marketing_real.db
(sqlite3.OperationalError) table faits_ventes already exists
[SQL: CREATE TABLE faits_ventes AS
  SELECT 
     InvoiceNo,
     StockCode,
     CustomerID,
     Quantity,
     UnitPrice,
     Date(InvoiceDate) AS DateVente
  FROM retail_brut]
(Background on this error at: https://sqlalche.me/e/14/e3q8)


In [21]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("ecom_marketing_real.db")

#Exportation en CSV
pd.read_sql_query("SELECT * FROM dim_clients;", conn).to_csv("dim_clients.csv", index=False, encoding = 'utf-8')
pd.read_sql_query("SELECT * FROM dim_produits;", conn).to_csv("dim_produits.csv", index=False, encoding = 'utf-8')
pd.read_sql_query("SELECT * FROM faits_ventes;", conn).to_csv("faits_ventes.csv", index=False, encoding = 'utf-8')

print("Super ! 'dim_clients.csv','dim_produits.csv' et 'faits_ventes.csv' sont prets pour Power BI")


Super ! 'dim_clients.csv','dim_produits.csv' et 'faits_ventes.csv' sont prets pour Power BI


In [22]:
%%sql
SELECT DISTINCT StockCode, Description
 FROM retail_brut
WHERE StockCode NOT GLOB '[0-9][0-9][0-9][0-9][0-9]*'
LIMIT 10

 * sqlite:///ecom_marketing_real.db
Done.


StockCode,Description
POST,POSTAGE
C2,CARRIAGE
M,Manual
BANK CHARGES,Bank Charges
PADS,PADS TO MATCH ALL CUSHIONS
DOT,DOTCOM POSTAGE


In [23]:
%%sql
DELETE FROM faits_ventes
WHERE StockCode NOT GLOB '[0-9][0-9][0-9][0-9][0-9]*'

 * sqlite:///ecom_marketing_real.db
0 rows affected.


[]

In [24]:
%%sql
DROP TABLE dim_produits;
CREATE TABLE dim_produits AS
SELECT DISTINCT StockCode, Description
FROM retail_brut
WHERE StockCode GLOB '[0-9][0-9][0-9][0-9][0-9]*'

 * sqlite:///ecom_marketing_real.db
Done.
Done.


[]

In [25]:
%%sql
SELECT CustomerID, CA, RANK() OVER(ORDER BY CA DESC) AS rang
FROM (
    SELECT CustomerID, SUM(Quantity*UnitPrice) AS CA 
    FROM retail_brut
    GROUP BY CustomerID
    )
LIMIT 20

 * sqlite:///ecom_marketing_real.db
Done.


CustomerID,CA,rang
14646.0,280206.02,1
18102.0,259657.3,2
17450.0,194390.79,3
16446.0,168472.5,4
14911.0,143711.17,5
12415.0,124914.53,6
14156.0,117210.08,7
17511.0,91062.38,8
16029.0,80850.84,9
12346.0,77183.6,10


In [26]:
%%sql
SELECT Mois, CA, LAG(CA,1) OVER (ORDER BY Mois) AS CA_mois_precedent, (CA - CA_mois_precedent) AS Croiss_CA
FROM (
    SELECT Mois, CA, LAG(CA,1) OVER (ORDER BY Mois) AS CA_mois_precedent
    FROM ( 
        SELECT strftime('%Y-%m', InvoiceDate) as Mois, 
               SUM(quantity*UnitPrice) AS CA 
        FROM retail_brut
        GROUP BY Mois
        )
)

 * sqlite:///ecom_marketing_real.db
Done.


Mois,CA,CA_mois_precedent,Croiss_CA
2010-12,570422.73,None,None
2011-01,568101.31,570422.73,-2321.4199999999255
2011-02,446084.92,568101.31,-122016.39000000007
2011-03,594081.76,446084.92,147996.84000000003
2011-04,468374.331,594081.76,-125707.429
2011-05,677355.15,468374.331,208980.81900000002
2011-06,660046.05,677355.15,-17309.099999999977
2011-07,598962.901,660046.05,-61083.14900000009
2011-08,644051.04,598962.901,45088.13900000008
2011-09,950690.202,644051.04,306639.162


In [29]:
%%sql
SELECT CustomerID, InvoiceDate 
FROM (
    SELECT CustomerID, InvoiceDate, ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY InvoiceDate) AS first_order
    FROM retail_brut
)
WHERE first_order = 1

 * sqlite:///ecom_marketing_real.db
Done.


CustomerID,InvoiceDate
12346.0,2011-01-18 10:01:00
12347.0,2010-12-07 14:57:00
12348.0,2010-12-16 19:09:00
12349.0,2011-11-21 09:51:00
12350.0,2011-02-02 16:01:00
12352.0,2011-02-16 12:33:00
12353.0,2011-05-19 17:47:00
12354.0,2011-04-21 13:11:00
12355.0,2011-05-09 13:49:00
12356.0,2011-01-18 09:50:00
